# Masked statistics under jit, no boolean indexing

In NumPy you might write `x[mask].mean()`. Under `jax.jit` that fails: the number of selected elements depends on the data, and compiled JAX code needs every shape known at trace time. The fix is to keep the full shape and let the mask decide what counts.

Write `masked_stats(x, mask)`, decorated with `@jax.jit` (keep the decorator from the starter):

- `x`: `(B, T)` float array.
- `mask`: `(B, T)` bool array. `True` marks a real entry.

Return a tuple of three `(B,)` arrays:

1. `mean`: float32 mean of the real entries in each row.
2. `maxv`: float32 maximum of the real entries in each row.
3. `count`: int32 number of real entries in each row.

A row with no real entries has `mean = 0.0`, `maxv = 0.0` and `count = 0`. No boolean indexing, no Python loops.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

@jax.jit
def masked_stats(x, mask):
    """x (B, T) float, mask (B, T) bool. Return (mean, maxv, count), each (B,)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 9.0], [4.0, -1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[true, true, false], [true, true, true]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [1.5, 1.0], "dtype": "float32"}, {"__nd__": [2.0, 4.0], "dtype": "float32"}, {"__nd__": [2, 3], "dtype": "int32"}]}}, {"a": [{"__nd__": [[5.0, 5.0], [-3.0, -7.0]], "dtype": "float64"}, {"__nd__": [[false, false], [false, true]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [0.0, -7.0], "dtype": "float32"}, {"__nd__": [0.0, -7.0], "dtype": "float32"}, {"__nd__": [0, 1], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-0.03419276725318417, -1.3597475403099617, -1.2247210785859324, -0.5103070767876675, -0.2979695111064471, -0.5273841930334252], [-0.5697263575719601, -0.056064439045617594, -0.7468856162565439, -1.8473247989741095, -1.5665487746995206, -0.09643216015562055], [-0.6803784532741461, -0.13656633397682774, -0.3790985670748533, -0.46311015859758675, -0.824513527530113, -0.20252987069345152]], "dtype": "float64"}, {"__nd__": [[false, false, false, true, false, true], [true, false, true, false, false, true], [true, true, true, false, true, true]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [-0.5188456773757935, -0.47101473808288574, -0.444617360830307], "dtype": "float32"}, {"__nd__": [-0.5103070735931396, -0.0964321568608284, -0.13656634092330933], "dtype": "float32"}, {"__nd__": [2, 3, 5], "dtype": "int32"}]}}, {"a": [{"__nd__": [[0.5446678079208929], [1.0428754765829538], [-0.20695643620832396], [-0.8135155419815723]], "dtype": "float64"}, {"__nd__": [[true], [false], [true], [false]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [0.5446677803993225, 0.0, -0.2069564312696457, 0.0], "dtype": "float32"}, {"__nd__": [0.5446677803993225, 0.0, -0.2069564312696457, 0.0], "dtype": "float32"}, {"__nd__": [1, 0, 1, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[0.3476505985155095, 0.24754574096284754, 1.0988127684144084, -1.284580778805345, -0.6616129303555477, -0.8381669607156745, -1.7340148462328515, 0.1264345551969962], [0.527804212495524, -0.7387900314758065, 1.3856470744961586, 0.8219243366604353, 0.6273764788355353, 0.4017070914409699, 0.955669564448635, -1.3319798395431022], [0.6139296582498643, 0.6027768335334479, -1.7677185771429749, 0.34703010205437973, -0.2504213467099684, 0.7815226960616993, -0.4390621876376686, -0.01824085764910033], [0.3428515173176555, -0.8762616887442077, 0.5985966481803844, -0.10496318852366823, 0.49248262367924284, -0.5218375063367878, 1.0862015432775176, 0.6052019784294742], [-0.17802502471933673, 0.6319571570936101, 1.259755161358625, 1.7911755134979888, -1.5735763704402195, 0.8831318116225195, 0.4650685085133813, -0.09386078018634399]], "dtype": "float64"}, {"__nd__": [[true, true, true, true, true, true, true, true], [true, true, true, true, true, true, true, true], [true, true, true, true, true, true, true, true], [true, true, true, true, true, true, true, true], [true, true, true, true, true, true, true, true]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [-0.3372415006160736, 0.33116984367370605, -0.016272952780127525, 0.2027839869260788, 0.3982032537460327], "dtype": "float32"}, {"__nd__": [1.0988128185272217, 1.3856470584869385, 0.7815226912498474, 1.086201548576355, 1.7911754846572876], "dtype": "float32"}, {"__nd__": [8, 8, 8, 8, 8], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-1.0066649349770713, 1.2571886134731436, -1.2617379934445705, 0.5669454657347489, 1.3018679962026896, -1.5996692880514796, -0.30251784048326236], [-1.3092168175162993, 0.24405410803590055, 1.5143751306746547, 2.0235560291721977, -1.7781144428835152, -0.5749490057210601, 0.7035450933309114]], "dtype": "float64"}, {"__nd__": [[false, false, false, false, false, false, false], [false, false, false, false, false, false, false]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [0.0, 0.0], "dtype": "float32"}, {"__nd__": [0.0, 0.0], "dtype": "float32"}, {"__nd__": [0, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[157.93726520932216, 42.12106442782976, -74.61519975907093, 29.71315096374385, -1.6619200980833249, -20.37407223881694, -73.44710985474204, 38.72597559430056, 30.787962636581117, -9.298412757635024], [-22.16879397006701, -128.4916570539673, -48.61756671414334, 120.64496713466424, -19.055825686602823, -143.96973168381254, 133.44428069513162, 53.02654084829202, 210.80870286253034, 6.2512169391719965], [-46.13845790516922, -144.76446987755466, 132.38383700638795, 256.95115129905116, -82.0934937223875, -64.70876402241984, 59.61887986723416, -83.03454354133866, -27.055836699218926, -34.98376994420694], [19.195888729205716, 109.48184434006801, 2.2067589713919693, 91.89100957801388, -41.988889600049355, 32.779853572340315, -213.8225589513189, -144.99480667813884, 79.59134126817742, -59.0149399040946], [57.99149234726574, 54.23442548146441, 132.22788582368145, 81.18590596762012, 101.69913501666113, -11.167133066420938, -69.82851765628781, -73.1558777725664, -48.80439402887327, -112.98291140131056], [-54.74435821203581, -9.256905150847414, 25.161195710581453, -33.889079006978534, -192.3682866353282, -7.228295450292372, 22.534578702721785, 108.4475594643844, 57.786389561582666, -64.35606861455095]], "dtype": "float64"}, {"__nd__": [[false, true, false, false, false, true, false, false, false, false], [false, true, false, true, false, false, true, true, false, false], [false, false, false, true, false, false, false, false, true, false], [false, false, false, false, false, false, true, false, false, false], [false, false, false, true, false, false, true, false, false, false], [false, true, false, false, false, false, true, false, false, true]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [10.873495101928711, 44.65603256225586, 114.9476547241211, -213.8225555419922, 5.6786956787109375, -17.026132583618164], "dtype": "float32"}, {"__nd__": [42.121063232421875, 133.44427490234375, 256.9511413574219, -213.8225555419922, 81.18590545654297, 22.534578323364258], "dtype": "float32"}, {"__nd__": [2, 4, 2, 1, 2, 3], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(masked_stats, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")